# **07 - Join shows + artists + venues into one table**
*** 

Joins the three cleaned tables into one row per show, checks the result for duplicates, and keeps only the shows we model: US shows with at least 3 listings. It keeps only the columns we plan to evaluate as model features.

1. Loads cleaned tables
2. Keeps only the chosen columns from each table (listed in the setup cell).
3. Joins `artists` onto `shows` on `artist_key` and `venues` onto `shows` on `venue_key` (many shows to one artist or venue, so the row count don't change).
4. Renames the two columns that clash (`country` exists in both artists and venues).
5. Checks the joined table for duplicates.
6. Filters to US shows, shows with at least `MIN_LISTINGS` listings, and prices up to `PRICE_CAP`.
7. Checks missing values per column and saves `data/joined_table.csv`.

**Input:** `data/shows_clean.csv`, `data/artists_clean.csv`, `data/venues_clean.csv`
**Output:** `data/joined_table.csv`, one row per show

## 1. Setup

In [20]:
# imports
import re
import numpy as np
import pandas as pd

# paths to the cleaned data
SHOWS_PATH = "../data/shows_clean.csv"
ARTISTS_PATH = "../data/artists_clean.csv"
VENUES_PATH = "../data/venues_clean.csv"
OUT_PATH = "../data/joined_table.csv"

KEEP_KEYS = True    # True = also keep artist_key and venue_key (needed for the train/test split by artist)

# Filtering decisions
KEEP_UNKNOWN_COUNTRY = False # True = keep shows whose venue country is still unknown after 06
MIN_LISTINGS = 3 # drop scarce markets
TARGET = "price_median" # or "price_min"
PRICE_CAP = 2000 # drop shows priced above this

# placeholder prices: a median above $1,000 that is more than 10x the cheapest ticket (replace with Hannah's idea)
PLACEHOLDER_MIN, PLACEHOLDER_RATIO = 1000, 10

# shows that aren't concerts to filter out
NON_CONCERT = re.compile(
    r"tape face|criss angel|cirque du soleil|dan and phil|- play$|buddy holly story|- musical"
    r"|emo nite|emo night|bop to the top|club 90s|gimme gimme disco|earlybirds club", re.I)

# Columns to keep from each table
SHOW_COLS = [
    "event_id", "name", "artist_name", "venue", # identifiers
    "start_utc", "start_local", "snapshot_date", # dates
    "weekday", "is_weekend", "month", "show_hour", "days_to_show", # date and time
    "listings", "max_listings", "n_priced_days", # supply
    "price_min", "price_median", "log_price_median", # target (pick one)
]
ARTIST_COLS = [
    "lastfm_name", # who Last.fm matched
    "listeners", "log_listeners", "playcount", "plays_per_listener", # popularity (Last.fm)
    "top_tag", "tags", # genre
    "on_tour", "n_shows", "n_venues", # touring
    "artist_type", "country", "year_formed", "years_active", # background
    "lastfm_status", "name_similarity", "low_confidence_match", # match quality
]
VENUE_COLS = [
    "city", "state", "country", "is_us", "lat", "lon", "timezone", # location
    "location_source", # where the location came from (06)
    "venue_tier", "median_max_listings", "venue_listings", # size (venue_listings is built in section 3)
    "sg_venue_score", "sg_upcoming_events", # SeatGeek variables
    "median_income", # county demographics from census data
]

## 2. Load the three tables

In [21]:
shows = pd.read_csv(SHOWS_PATH, low_memory=False, parse_dates=["start_utc", "start_local", "snapshot_date"])
artists = pd.read_csv(ARTISTS_PATH, low_memory=False)
venues = pd.read_csv(VENUES_PATH, low_memory=False)

In [22]:
# making sure ids/keys are unique before attempting to merge 
assert shows["event_id"].is_unique, "shows_clean.csv has repeated event_id"
assert artists["artist_key"].is_unique, "artists_clean.csv has repeated artist_key"
assert venues["venue_key"].is_unique, "venues_clean.csv has repeated venue_key"

## 3. Picking the columns
`country` is in both `artists` (where the artist is from) and `venues` (where the show is), so they are renamed `artist_country` and `venue_country`. The artists' `n_shows` and `n_venues` count how many shows and venues that artist has in our data.

Revisit this with Miriam because we are using venue's median listing price before train/test split so this could technically be data leakage even though we are not leaking the target variable. Lets see what she says. CHECK ???

In [23]:
# Creating venue size stand-in from venue's median listings on snapshot day across ALL its shows
# (unlike median_max_listings, it only uses what was known on the snapshot day)
venues["venue_listings"] = venues["venue_key"].map(shows.groupby("venue_key")["listings"].median())

# adding in the keys so we can merge (and renaming columns to avoid dup col names)
s = shows[["artist_key", "venue_key"] + SHOW_COLS]
a = artists[["artist_key"] + ARTIST_COLS].rename(columns={"country": "artist_country"})
v = venues[["venue_key"] + VENUE_COLS].rename(columns={"country": "venue_country"})

## 4. Join

In [24]:
# merging the data together
joined = (s.merge(a, on="artist_key", how="left", validate="many_to_one")
           .merge(v, on="venue_key", how="left", validate="many_to_one"))

# checking that the len is the same since we should have the name num shows before/after
# (merging into shows so should not change)
assert len(joined) == len(shows), "join changed the number of rows"

# looking at the merge results
print(joined.shape)

# looking at variable population after merge
print(f"shows with an artist row: {joined['listeners'].notna().mean():.0%} have Last.fm popularity")
print(f"shows with a venue row:   {joined['venue_tier'].notna().mean():.0%} have a venue tier")
# these look good

(29587, 51)
shows with an artist row: 94% have Last.fm popularity
shows with a venue row:   100% have a venue tier


## 5. Checking for duplicates after the join
06 already removed duplicates in each table. These checks make sure the join did not add any: each `event_id` appears once, and no two rows are the same artist at the same venue at the same local start time.

In [25]:
# ensuring each event is still unique
assert joined["event_id"].is_unique, "the join repeated an event_id"

# defining the key of each show (what defines a show uniquely)
KEY = ["artist_key", "venue_key", "start_local"]
# checking for show dupes
same_show = joined.duplicated(KEY, keep=False)
print(f"Rows that are the same artist, venue and start time: {same_show.sum()}")
print(f"{len(joined):,} rows after the duplicate check")
# so this matches shows length from before, telling the merge worked correctly

Rows that are the same artist, venue and start time: 0
29,587 rows after the duplicate check


## 6. Filtering
All row filters are applied here, in order, and the table at the end shows how many shows each one removes.
- **US only:** `is_us` comes from the venue's country (SeatGeek, or filled in by hand in 06). Venues whose country is still unknown are dropped unless `KEEP_UNKNOWN_COUNTRY = True`.
- **Date doesn't match the snapshot:** the show date (UTC) should equal `snapshot_date + days_to_show`. A few don't (probably rescheduled shows; 03 keeps the first start time it saw).
- **Not a concert:** names matching `NON_CONCERT` (magic, circus, plays, musicals, party nights), and artists whose Last.fm tags say comedy or stand-up.
- **3+ listings:** shows with fewer than `MIN_LISTINGS` listings at the snapshot are thin markets where one seller sets the price.
- **Extreme or placeholder price:** `TARGET` above `PRICE_CAP`, or a median above $1,000 that is more than 10 times the cheapest ticket (resellers' placeholder listings, often the same odd price at unrelated shows). (this one should change by Hannah's recommendation for mathematically identifying an outlier)
- **No Last.fm popularity:** popularity is the main thing we expect to drive price so we require it.

In [26]:
# flagging shows with a date gap (days till show and snapshot date don't align)
date_gap = (joined["start_utc"].dt.normalize() - joined["snapshot_date"]).dt.days - joined["days_to_show"]
# flagging comedy shows
is_comedy = joined["tags"].str.contains(r"comedy|stand-up", case=False, na=False)
# flagging us shows
is_us = joined["is_us"].astype(str).eq("True")

# flagging the problematic rows we want to filter out
flags = {
    "not a US show": ~(is_us | (joined["is_us"].isna() & KEEP_UNKNOWN_COUNTRY)),
    "date doesn't match the snapshot": date_gap != 0,
    "not a concert": joined["name"].str.contains(NON_CONCERT, na=False) | is_comedy,
    f"fewer than {MIN_LISTINGS} listings": joined["listings"] < MIN_LISTINGS,
    "extreme or placeholder price": (joined[TARGET] > PRICE_CAP)
        | ((joined["price_median"] > PLACEHOLDER_MIN) & (joined["price_median"] > PLACEHOLDER_RATIO * joined["price_min"])),
    "no Last.fm popularity": joined["listeners"].isna()}

# looking at the country of shows we are dropping
print("Dropped as non-US (by venue country):")
print(joined.loc[flags["not a US show"], "venue_country"].fillna("unknown").value_counts().head(10).to_string())
# these look correct

# printing the most common non-concert shows to make sure this worked
print("\nMost common acts dropped as not concerts:")
print(joined.loc[flags["not a concert"], "artist_name"].value_counts().head(10).to_string())
# these are all non-concerts so it worked

# making an attrition table to see how we lose shows as we filter them out (help of AI)
keep = pd.Series(True, index=joined.index)
funnel = [("shows after the join", len(joined))]
for label, flag in flags.items():
    keep &= ~flag
    funnel.append((f"minus {label}", int(keep.sum())))
joined = joined[keep].reset_index(drop=True)

display(pd.DataFrame(funnel, columns=["step", "shows left"]).assign(removed=lambda d: -d["shows left"].diff()))

Dropped as non-US (by venue country):
venue_country
Canada         1319
UK              316
unknown         194
Germany          60
France           34
Spain            33
Netherlands      26
Ireland          19
Puerto Rico      19
Australia        14

Most common acts dropped as not concerts:
artist_name
Tape Face                     76
Cirque du Soleil Mad Apple    59
Criss Angel                   57
Midnight                      55
Bop To The Top                53
Buddy                         46
Club 90s                      43
Earlybirds Club               40
Emo Night Brooklyn            40
Gimme Gimme Disco             38


,step,shows left,removed
0,shows after the join,29587,NaN
1,minus not a US show,27477,2110.0
2,minus date doesn't match the snapshot,27470,7.0
3,minus not a concert,26660,810.0
4,minus fewer than 3 listings,16117,10543.0
5,minus extreme or placeholder price,16055,62.0
6,minus no Last.fm popularity,15084,971.0


## 7. Check missing values and save

In [27]:
# checking if we want to filter the columns (we do and set to True)
if not KEEP_KEYS:
    # dropping the shared keys
    joined = joined.drop(columns=["artist_key", "venue_key"])

# looking at the missing data
missing = (joined.isna().mean().mul(100).round(1) # computing missing percentages
                 .rename("% missing").to_frame()) # formatting in table 
# looking at the table of missing values
display(missing.sort_values("% missing", ascending=False))

,% missing
years_active,88.4
year_formed,88.4
artist_country,86.8
median_income,21.2
sg_upcoming_events,20.9
sg_venue_score,20.9
lon,20.9
lat,20.9
tags,9.4
top_tag,9.4


## 8. Exporting Cleaned/Merged Data

In [28]:
# exporting the 
joined.to_csv(OUT_PATH, index=False)

# looking at the new data ready to be worked with
joined.head()

,artist_key,venue_key,event_id,name,artist_name,venue,start_utc,start_local,snapshot_date,weekday,...,lat,lon,timezone,location_source,venue_tier,median_max_listings,venue_listings,sg_venue_score,sg_upcoming_events,median_income
0,houndmouth,frederik meijer gardens amphitheater|grand rapids,69d8115d8f7499b0f6720c9c,Houndmouth,Houndmouth,Frederik Meijer Gardens Amphitheater,2026-07-28,2026-07-27 20:00:00,2026-07-18,Monday,...,42.9795,-85.5901,America/Detroit,seatgeek,unknown,12.0,7.0,0.52,0.0,80390.0
1,5 seconds of summer,riverbend music center|cincinnati,68fa3930a768106fd704bfa3,5 Seconds of Summer,5 Seconds of Summer,Riverbend Music Center,2026-07-28,2026-07-27 20:00:00,2026-07-18,Monday,...,39.0520,-84.4170,America/New_York,seatgeek,"large (2,500+)",25.0,21.0,0.74,0.0,70816.0
2,poppy,admiral|omaha,69950bd9ee953753416d5dcd,Poppy,Poppy,The Admiral,2026-07-28,2026-07-27 19:00:00,2026-07-18,Monday,...,NaN,NaN,America/Chicago,seatgeek_city,unknown,5.0,2.0,NaN,NaN,NaN
3,melanie martinez,spectrum center|charlotte,69e0ff64207f703d14f2e7e3,Melanie Martinez,Melanie Martinez,Spectrum Center,2026-07-28,2026-07-27 20:00:00,2026-07-18,Monday,...,NaN,NaN,America/New_York,seatgeek_city,unknown,114.0,91.0,NaN,NaN,NaN
4,toast,ames center|burnsville,69e7d17bdda6ab39dfce4e7b,Toast - The Best of Bread,Toast,Ames Center,2026-07-28,2026-07-27 19:00:00,2026-07-18,Monday,...,44.7744,-93.2792,America/Chicago,seatgeek,unknown,60.0,60.0,0.46,20.0,105212.0
